# Margo Engine for data science

Million-point scatter plots, volumes, isosurfaces and surfaces from NumPy and pandas, drawn by Margo Engine with WebGPU in the browser. Arrays go to the browser as binary float32 buffers through the Jupyter widget protocol, not as JSON.

Setup (once, from the Margo Engine repository): `npm run build:packages`, `node python/build-frontend.mjs`, `pip install ./python`, then open this notebook in JupyterLab 4 or Notebook 7 with ipywidgets 8, in a WebGPU browser.

In [ ]:
import numpy as np
import pandas as pd
import margoengine as margo

rng = np.random.default_rng(22)

## A million points from a DataFrame

Five Gaussian clusters, coloured by each point's distance from its cluster centre. The point budget draws an even sample each frame, so orbiting stays smooth.

In [ ]:
n = 1_000_000
centres = np.array([[0, 0, 0], [4, 1, 2], [-3, 3, 1], [1, -4, 3], [-2, -2, -2]], dtype=np.float32)
spread = np.array([1.0, 0.6, 0.8, 0.5, 0.7], dtype=np.float32)
cluster = rng.integers(0, len(centres), n)
offset = rng.standard_normal((n, 3)).astype(np.float32) * spread[cluster, None]
points = centres[cluster] + offset
df = pd.DataFrame({"x": points[:, 0], "y": points[:, 1], "z": points[:, 2], "sigma": np.linalg.norm(offset, axis=1) / spread[cluster], "cluster": cluster})
margo.scatter(df, x="x", y="y", z="z", color="sigma", colormap="aurora", range=(0, 3))

## A volume and its isosurface

A gyroid on a 96³ grid (`values[z][y][x]`): ray-marched with a transfer function, then as a surface extracted by marching cubes on the GPU.

In [ ]:
g = np.linspace(-np.pi, np.pi, 96, dtype=np.float32)
X, Y, Z = np.meshgrid(g, g, g, indexing="xy")
gyroid = np.sin(X) * np.cos(Y) + np.sin(Y) * np.cos(Z) + np.sin(Z) * np.cos(X)
margo.volume(gyroid, colormap="ember", opacity=[(0, 0), (0.55, 0), (0.75, 0.4), (1, 0.9)], bounds=(-np.pi, np.pi) * 3)

In [ ]:
margo.isosurface(gyroid, 0.0, bounds=(-np.pi, np.pi) * 3, color="#9cc9ff")

## A surface plot

Heights over a grid (`z[row][column]`), coloured through a colour map with contour lines and a colour bar.

In [ ]:
x = np.linspace(-6, 6, 256, dtype=np.float32)
xx, yy = np.meshgrid(x, x)
r = np.hypot(xx + 2, yy - 1)
heights = np.cos(2 * r) * np.exp(-0.18 * r) + 0.8 * np.cos(2.6 * np.hypot(xx - 2.5, yy + 1.5)) * np.exp(-0.22 * np.hypot(xx - 2.5, yy + 1.5))
margo.surface(heights, x=(-6, 6), y=(-6, 6), colormap="coolwarm", contours=12)

## Several layers, then a page to share

A figure takes any mix of layers. `save_html` writes one self-contained file that opens in any WebGPU browser, no server needed.

In [ ]:
t = np.linspace(0, 8 * np.pi, 2000, dtype=np.float32)
fig = margo.figure(theme="light", titles={"x": "x", "y": "y", "z": "height"})
fig.surface(heights[::4, ::4], x=(-6, 6), y=(-6, 6), colormap="glacier", contours=8)
fig.line(x=5 * np.cos(t) * t / t.max(), y=5 * np.sin(t) * t / t.max(), z=1.5 + t / t.max(), color=t, colormap="ember", width=3)
fig.save_html("margo-figure.html")
fig